# Normative Anomaly Detection & Age-Associated Trajectory Analysis


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from statsmodels.stats.multitest import multipletests

def cliffs_delta(lst1, lst2):
    n1, n2 = len(lst1), len(lst2)
    greater = sum(x > y for x in lst1 for y in lst2)
    less = sum(x < y for x in lst1 for y in lst2)
    return (greater - less) / (n1 * n2)

df_feat = pd.read_csv(os.path.join('data', 'monash_14d_features.csv'))

combined_cols = [
    'semg_rms', 'semg_std', 'semg_p25', 'semg_p75', 'semg_p95',
    'angle_mean', 'angle_std', 'angle_range', 'angle_p25', 'angle_p75', 'angle_p95',
    'imu_accel_rms', 'imu_accel_std', 'imu_accel_p95'
]

young_subjs = df_feat[df_feat['cohort'] == 'Young Normative']['subject_id'].unique()
n_resamples = 100
val_burdens, mid_burdens, old_burdens = [], [], []

for seed in range(n_resamples):
    np.random.seed(seed)
    shuffled = np.random.permutation(young_subjs)
    tr_subjs, val_subjs = shuffled[:18], shuffled[18:]
    df_tr = df_feat[df_feat['subject_id'].isin(tr_subjs)]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(df_tr[combined_cols])
    iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=seed)
    iso.fit(X_tr)
    tau = np.percentile(iso.score_samples(X_tr), 5)
    X_all = scaler.transform(df_feat[combined_cols])
    scores = iso.score_samples(X_all)
    df_temp = df_feat.copy()
    df_temp['flag'] = (scores < tau).astype(int)
    subj_b = df_temp.groupby(['subject_id', 'cohort'])['flag'].mean().reset_index()
    val_burdens.append(subj_b[subj_b['subject_id'].isin(val_subjs)]['flag'].mean() * 100.0)
    mid_burdens.append(subj_b[subj_b['cohort'] == 'Middle Adult']['flag'].mean() * 100.0)
    old_burdens.append(subj_b[subj_b['cohort'] == 'Older Adult']['flag'].mean() * 100.0)

print(f'Held-Out Young Outlier Burden: {np.mean(val_burdens):.2f}%')
print(f'Middle Cohort Outlier Burden: {np.mean(mid_burdens):.2f}%')
print(f'Older Cohort Outlier Burden:  {np.mean(old_burdens):.2f}%')

scaler_y = StandardScaler()
young_df = df_feat[df_feat['cohort'] == 'Young Normative']
scaler_y.fit(young_df[combined_cols])
iso_full = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
iso_full.fit(scaler_y.transform(young_df[combined_cols]))

df_feat['normality_score'] = iso_full.score_samples(scaler_y.transform(df_feat[combined_cols]))
lmm_adj = smf.mixedlm('normality_score ~ age + sex + weight + height', df_feat, groups=df_feat['subject_id']).fit()
print(f'Age Beta = {lmm_adj.params["age"]:.6f}, p = {lmm_adj.pvalues["age"]:.4f}')
